In [1]:
import jax.numpy as jnp
import numpy as np

In [2]:
n1, n2, n3 = 100, 200, 300
n = jnp.stack([n1, n2, n3])
x1 = np.random.normal(0, 1, size=(10, n1))
x2 = np.random.normal(0, 5, size=(10, n2))
x3 = np.random.normal(5, 5, size=(10, n2))
mu = jnp.stack([jnp.mean(x1, axis=1),jnp.mean(x3, axis=1),jnp.mean(x3, axis=1),])
cov = jnp.stack([jnp.cov(x1), jnp.cov(x2), jnp.cov(x3)])

c_true = jnp.cov(jnp.concat([x1, x2, x3], axis=-1))

def aggregate_covariance(covs, sample_sizes, means):
    #* Join covariances by algorithm found here:
    #* https://stats.stackexchange.com/questions/655028/combining-mean-and-covariance-matrix-of-two-populations#:~:text=Your%20problem:%20The%20case%20you,%CB%89x2)T%5D.
    cov_net = covs[0]
    mu_net = means[0]
    n_net = sample_sizes[0]
    
    for i in range(1, len(covs)):
        cov = covs[i]
        mu = means[i]
        n = sample_sizes[i]

        delta_mu = mu - mu_net
        delta_prod=jnp.outer(delta_mu, delta_mu)
        
        cov_net = 1/(n_net+n-1) * ((n_net-1)*cov_net + (n-1)*cov + (n_net*n)/(n_net+n) * delta_prod)
        mu_net = (n_net * mu_net + n*mu)/(n_net+n)
        n_net = n_net + n
    return cov_net, mu_net

def aggregate_covariance_new(covs, sample_sizes, means):
    n = jnp.sum(sample_sizes)
    pooled_mean = np.sum(sample_sizes[:, np.newaxis]*means, axis=0)/n

    delta_sum = jnp.sum(sample_sizes[:, None, None]*jnp.einsum('ij,ik->ijk', means, means), axis=0)
    pooled_mean_prod = jnp.outer(pooled_mean, pooled_mean)

    cov_net = 1/(n-1) *(jnp.sum((sample_sizes[:, None, None]-1)*covs, axis=0) + delta_sum- n*pooled_mean_prod)

    return cov_net, pooled_mean


def aggregate_m2(m2s, sample_sizes, means):
    n = jnp.sum(sample_sizes)
    pooled_mean = jnp.sum(sample_sizes[:, np.newaxis]*means, axis=0)/n

    delta_sum = jnp.sum(sample_sizes[:, None, None]*jnp.einsum('ij,ik->ijk', means, means), axis=0)
    pooled_mean_prod = jnp.outer(pooled_mean, pooled_mean)

    m2_net = jnp.sum(m2s, axis=0) + delta_sum- n*pooled_mean_prod

    return m2_net, pooled_mean


def aggregate_covariance_agg(covs, sample_sizes, means):
    #* Join covariances by algorithm found here:
    #* https://stats.stackexchange.com/questions/655028/combining-mean-and-covariance-matrix-of-two-populations#:~:text=Your%20problem:%20The%20case%20you,%CB%89x2)T%5D.
    n = jnp.sum(sample_sizes)
    m2s = sample_sizes[:, None, None] * covs
    m2_net, pooled_mean = aggregate_m2(m2s, sample_sizes, means)
    
    cov_net = 1/(n-1) * m2_net

    return cov_net, pooled_mean

c1, m1 = aggregate_covariance(cov, n, mu)
c2, m2 = aggregate_covariance_new(cov, n, mu)
c3, m3 = aggregate_covariance_agg(cov, n, mu)

In [3]:
print(jnp.diag((c1-c_true)/c_true))
print(jnp.diag((c2-c_true)/c_true))
print(jnp.diag((c3-c_true)/c_true))


[-0.10414122 -0.00751216 -0.06204071 -0.03182992 -0.05556056 -0.04546967
 -0.075296   -0.06022203 -0.0708575  -0.05111112]
[-0.10414109 -0.007512   -0.0620405  -0.03182992 -0.05556056 -0.04546982
 -0.075296   -0.06022203 -0.0708575  -0.05111104]
[-0.10096062 -0.00390975 -0.05891992 -0.02839727 -0.05221969 -0.04209402
 -0.07221323 -0.05688903 -0.06757782 -0.04782214]
